# SNR reproduction: full-protocol runs on Colab (GPU)

This notebook is a thin wrapper. All logic lives in the repository (`src/`, `scripts/`), and the notebook only installs, prepares data, tests, and launches the sweeps.

* **Local CPU (done):** the reduced protocol in `configs/experiments/*.yaml`. PM uses 600 tasks; PS uses 50 tasks x 50 epochs.
* **Here:** the paper protocol. PM uses 2400 tasks and 5 seeds; PS uses 500 tasks x 100 epochs. Set the overrides below.
* **Resuming:** runs checkpoint to Google Drive every N tasks and resume after a disconnect (`--resume`).
* **Tracking:** pick `trackio` (the dashboard the deck cites) or `wandb` (the course's public project). Put credentials in Colab *Secrets*, never in a cell.

In [ ]:
# Runtime > Change runtime type > GPU. Check what we got.
!nvidia-smi -L || echo "no GPU: runs will use the CPU"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUN_DIR = '/content/drive/MyDrive/snr-reproduction'   # checkpoints + metric CSVs survive disconnects

In [ ]:
%cd /content
!git clone -q -b milestone2-core-scope https://github.com/CVTrung/SNR-Continual-Learning-Reproduction.git repo || (cd repo && git pull -q)
%cd /content/repo
!git log --oneline -1
# Colab already has torch/numpy/pandas/matplotlib; install only what is missing.
!pip -q install tiktoken==0.14.0 huggingface_hub pyyaml tabulate wandb trackio

In [ ]:
# Programmatic data preparation (Hugging Face MNIST + pinned Shakespeare text), hash-verified.
!python -m scripts.prepare_data
!python -m pytest -q

In [ ]:
# Tracking credentials from Colab Secrets (left panel, key icon). Leave a secret empty to skip it.
import os
from google.colab import userdata
for key in ('WANDB_API_KEY', 'HF_TOKEN', 'TRACKIO_SPACE_ID'):
    try:
        os.environ[key] = userdata.get(key)
    except Exception:
        pass
TRACKING = 'trackio' if os.environ.get('TRACKIO_SPACE_ID') else ('wandb' if os.environ.get('WANDB_API_KEY') else 'none')
print('tracking backend:', TRACKING)

## Full-protocol overrides

`scripts/sweep.py` reads the same YAML files as the local runs. The cell below writes full-protocol copies to `/content/full_*.yaml`, with output and checkpoints on Drive. On a GPU, single-process throughput matters more than parallel workers, so start with `--workers 2` and raise it while GPU utilisation is low.

In [ ]:
import yaml, pathlib
def full(src, dst, **common):
    spec = yaml.safe_load(open(src))
    spec['common'].update(common, out=f'{RUN_DIR}/metrics', checkpoint_dir=f'{RUN_DIR}/checkpoints',
                          checkpoint_every=25, resume=True, tracking=TRACKING, device='cuda')
    pathlib.Path(dst).write_text(yaml.safe_dump(spec, sort_keys=False))
    print(dst, spec['common'])
full('configs/experiments/main.yaml', '/content/full_main.yaml', tasks=2400)
spec = yaml.safe_load(open('/content/full_main.yaml')); spec['seeds'] = [100, 101, 102, 103, 104]
pathlib.Path('/content/full_main.yaml').write_text(yaml.safe_dump(spec, sort_keys=False))
full('configs/experiments/shakespeare.yaml', '/content/full_shakespeare.yaml', tasks=500, epochs=100)
full('configs/experiments/sensitivity.yaml', '/content/full_sensitivity.yaml', tasks=2400)

In [ ]:
# Measure before committing hours: 3 tasks of each benchmark, then extrapolate.
!python -m scripts.run_experiments --benchmark pm --reset snr --eta 0.005 --tasks 3 --device cuda --out /content/smoke
!python -m scripts.run_experiments --benchmark ps --reset snr --penalty l2 --lam 1e-4 --eta 0.001 --tasks 1 --epochs 5 --device cuda --out /content/smoke

In [ ]:
# Launch (re-run this cell after a disconnect: finished runs are skipped, unfinished ones resume).
!python -m scripts.sweep /content/full_shakespeare.yaml --workers 2

In [ ]:
!python -m scripts.sweep /content/full_main.yaml /content/full_sensitivity.yaml --workers 3

In [ ]:
# Summaries and figures from the Drive copy of the metrics.
!rm -rf results/metrics && ln -s {RUN_DIR}/metrics results/metrics
!python -m scripts.summarize_results && python -m scripts.plot_results
!cat results/tables/pm_main.md results/tables/ps_main.md